In [1]:
#In this challenge, we ask you to build a predictive model that answers the question:“what sorts of people were more likely to survive?”
#using passenger data (ie name, age, gender, socio-economic class, etc).

#We highly recommend Alexis Cook’s Titanic Tutorial: https://www.kaggle.com/code/alexisbcook/titanic-tutorial
#and this starter notebook : https://www.kaggle.com/code/gusthema/titanic-competition-w-tensorflow-decision-forests


# ========================= Data Dictionary =========================
# survival : Passenger survival status
#            0 = Did not survive, 1 = Survived
#
# pclass   : Ticket class
#            1 = First class
#            2 = Second class
#            3 = Third class
#
# sex      : Passenger's gender
#
# age      : Passenger's age (in years)
#
# sibsp    : Number of siblings/spouses aboard the Titanic
#
# parch    : Number of parents/children aboard the Titanic
#
# ticket   : Ticket number
#
# fare     : Ticket fare paid by the passenger
#
# cabin    : Cabin number
#
# embarked : Port of embarkation
#            C = Cherbourg
#            Q = Queenstown
#            S = Southampton
# ================================================================


# ========================= Variable Notes =========================
#
# pclass : A proxy for socio-economic status (SES)
#          1st = Upper class
#          2nd = Middle class
#          3rd = Lower class
#
# age    : Age is fractional if less than 1 year old.
#          If the age was estimated, it is recorded as xx.5.
#
# sibsp  : Number of siblings and/or spouses aboard the Titanic.
#          Family relationships are defined as:
#          - Sibling: brother, sister, stepbrother, stepsister
#          - Spouse : husband, wife
#            (Mistresses and fiancés are not included.)
#
# parch  : Number of parents and/or children aboard the Titanic.
#          Family relationships are defined as:
#          - Parent: mother, father
#          - Child : daughter, son, stepdaughter, stepson
#          Some children traveled only with a nanny,
#          so their parch value is 0.
#
# ================================================================

In [2]:
import pandas as pd
df = pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [4]:
df.shape

(891, 12)

In [5]:
#قبل از ساخت مدل میخوایم داده ها رو بشناسیم به این کار میگن EDA
#Exploratory Data Analysis
#یعنی: تحلیل اکتشافی داده‌ها
#هدف EDA این است که قبل از ساخت مدل، داده را بشناسیم و بفهمیم:
#چه ویژگی‌هایی داریم؟
#داده‌ها چه شکلی هستند؟
#چه مشکلاتی دارند؟
#کدام فیچرها احتمالاً مهم‌تر هستند؟
#چه الگوهایی داخل داده وجود دارد؟

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [7]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [8]:
# ========================= describe() Interpretation =========================
#
# PassengerId: 
# count = 891  → تعداد کل رکوردهای مسافران در دیتاست
# mean = 446   → میانگین شماره شناسایی مسافران (از نظر آماری اهمیت خاصی ندارد)
# std = 257.35 → پراکندگی شماره شناسایی‌ها
# min = 1      → اولین PassengerId
# max = 891    → آخرین PassengerId
#
# ---------------------------------------------------------------------------
#
# Survived:
# count = 891 → تعداد مسافرانی که مقدار Survival آن‌ها مشخص است
# mean = 0.3838 → نسبت میانگین زنده‌ماندگان
#               یعنی حدود 38.38٪ از مسافران زنده مانده‌اند.
#               چون Survived یک ستون باینری است:
#               1 = فوت کرده= 0،  زنده مانده
#               میانگین آن در واقع برابر با میانگین 1 ها ینی درصد افراد زنده‌مانده است.
#
# std = 0.4865 → میزان پراکندگی مقادیر 0 و 1
#
# min = 0      → کمترین مقدار (فوت شده)
# max = 1      → بیشترین مقدار (زنده مانده)
#
# 25%, 50%, 75%:
# نشان می‌دهند توزیع داده بین صفر و یک چگونه است.
# Median (50%) = 0 یعنی حداقل نیمی از افراد فوت کرده‌اند.
#
# ---------------------------------------------------------------------------
#
# Pclass:
# count = 891 → برای همه مسافران کلاس بلیت مشخص است.
# mean = 2.308 → میانگین کلاس مسافران حدود 2.3 است.
#               یعنی بیشتر مسافران در کلاس 2 و 3 بوده‌اند.
#
# std = 0.836 → میزان پراکندگی کلاس‌ها
#
# min = 1      → کلاس اول (ثروتمندتر)
# max = 3      → کلاس سوم (اقتصادی‌تر)
#
# 50% = 3      → حداقل نیمی از مسافران در کلاس سوم بوده‌اند.
#
# ---------------------------------------------------------------------------
#
# Age:
# count = 714 → فقط سن 714 نفر ثبت شده است.
#               پس 177 مقدار Age گمشده داریم.
#
# mean = 29.69 → میانگین سن مسافران حدود 30 سال بوده است.
#
# std = 14.52 → پراکندگی سن‌ها حدود 14.5 سال است.
#
# min = 0.42  → کمترین سن ثبت شده حدود 5 ماه است.
#
# 25% = 20.12 → 25 درصد مسافران کمتر از حدود 20 سال سن داشتند.
#
# 50% = 28    → سن میانه مسافران 28 سال است.
#
# 75% = 38    → 75 درصد مسافران کمتر از 38 سال بودند.
#
# max = 80    → مسن‌ترین مسافر 80 سال داشته است.
#
# ---------------------------------------------------------------------------
#
# SibSp:
# (Siblings + Spouses)
# تعداد خواهر، برادر یا همسر همراه مسافر
#
# mean = 0.523 → به طور متوسط هر مسافر حدود نیم نفر همراه از این نوع داشته.
#
# min = 0      → بعضی مسافران هیچ خواهر، برادر یا همسری همراه نداشتند.
#
# max = 8      → بیشترین مقدار همراه از این نوع 8 نفر بوده است.
#
# 50% = 0      → حداقل نیمی از مسافران هیچ خواهر، برادر یا همسری همراه نداشتند.
#
# ---------------------------------------------------------------------------
#
# Parch:
# (Parents + Children)
# تعداد والدین یا فرزندان همراه مسافر
#
# mean = 0.381 → به طور متوسط کمتر از یک والد یا فرزند همراه وجود داشته.
#
# min = 0      → بدون والد یا فرزند همراه
#
# max = 6      → بیشترین تعداد والد/فرزند همراه 6 نفر بوده است.
#
# 50% = 0      → حداقل نیمی از مسافران والد یا فرزند همراه نداشتند.
#
# ---------------------------------------------------------------------------
#
# Fare:
# مبلغ بلیت پرداختی
#
# mean = 32.20 → میانگین قیمت بلیت حدود 32 واحد پولی بوده است.
#
# std = 49.69  → پراکندگی قیمت بلیت زیاد است.
#                یعنی اختلاف قیمت بلیت‌ها زیاد بوده.
#
# min = 0      → بعضی رکوردها Fare صفر دارند.
#
# 50% = 14.45  → نیمی از مسافران کمتر از حدود 14.45 پرداخت کرده‌اند.
#
# 75% = 31     → 75 درصد مسافران کمتر از 31 پرداخت کرده‌اند.
#
# max = 512.32 → گران‌ترین بلیت حدود 512 بوده است.
#
# ===========================================================================
#نکات مهم
# Survived.mean() = 0.3838 → دیتاست نامتوازن شدید نیست، ولی کلاس‌ها برابر هم نیستند (حدود 38٪ بقا).
# Age.count() = 714 → باید برای Age تصمیم بگیریم با 177 مقدار خالی چه کنیم.
# Fare.std() خیلی بزرگ‌تر از میانگین است → احتمالاً چند مقدار خیلی بزرگ (outlier) وجود دارد.
# PassengerId احتمالاً ویژگی مفیدی نیست و بعداً بررسی می‌کنیم که حذف شود یا نه.

In [9]:
#پانداس فقط ستون‌های عددی را خلاصه می‌کند و اطلاعات آماری مربوط به فقط ستون های عددی  را نشان می‌دهد:
#برای ستون هایی که نوع داده شان object ینی متن یا رشته است از کد زیر استفاده میکنیم
df.describe(include=["object", "string"])

,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


In [10]:
# ================= خلاصه ستون‌های متنی (Categorical Features) =================
#
# معنی پارامترها:
# count  : تعداد مقادیر غیرگمشده در ستون
# unique : تعداد مقادیر متفاوت و یکتا در ستون
# top    : پرتکرارترین مقدار در ستون
# freq   : تعداد تکرار مقدار top
#
# ----------------------------------------------------------------
#
# Name
# - count = 891
#   همه‌ی 891 مسافر دارای نام ثبت‌شده هستند.
#
# - unique = 891
#   هر مسافر نام متفاوتی دارد و هیچ نامی تکرار نشده است.
#
# - top = Braund, Mr. Owen Harris
#   یکی از نام‌های موجود در دیتاست است.
#   چون همه نام‌ها فقط یک بار آمده‌اند، این مقدار معنی خاصی ندارد.
#
# - freq = 1
#   این نام فقط یک بار در دیتاست وجود دارد.
#
# ----------------------------------------------------------------
#
# Sex
# - count = 891
#   هیچ مقدار گمشده‌ای در جنسیت وجود ندارد.
#
# - unique = 2
#   فقط دو مقدار متفاوت وجود دارد:
#   male و female
#
# - top = male
#   پرتکرارترین مقدار جنسیت، مرد است.
#
# - freq = 577
#   تعداد مردان در دیتاست 577 نفر است.
#
# ----------------------------------------------------------------
#
# Ticket
# - count = 891
#   همه‌ی مسافران شماره بلیت دارند.
#
# - unique = 681
#   681 شماره بلیت متفاوت وجود دارد.
#   یعنی بعضی مسافران شماره بلیت مشترک دارند.
#
# - top = 347082
#   پرتکرارترین شماره بلیت در دیتاست است.
#
# - freq = 7
#   شماره بلیت 347082 برای 7 نفر استفاده شده است.
#
# ----------------------------------------------------------------
#
# Cabin
# - count = 204
#   فقط 204 مسافر شماره کابین دارند.
#   بقیه‌ی مقادیر گمشده هستند.
#
# - unique = 147
#   147 شماره کابین متفاوت ثبت شده است.
#
# - top = G6
#   پرتکرارترین شماره کابین است.
#
# - freq = 4
#   کابین G6 برای 4 مسافر ثبت شده است.
#
# ----------------------------------------------------------------
#
# Embarked
# - count = 889
#   از 891 مسافر، اطلاعات بندر سوار شدن 889 نفر موجود است.
#   2 مقدار گمشده وجود دارد.
#
# - unique = 3
#   سه مقدار متفاوت وجود دارد:
#   C = Cherbourg
#   Q = Queenstown
#   S = Southampton
#
# - top = S
#   پرتکرارترین بندر سوار شدن Southampton است.
#
# - freq = 644
#   تعداد مسافرانی که از Southampton سوار شدند 644 نفر است.
#
# ============================================================================
 
# ================= خلاصه نکات مهم ستون‌های متنی (Categorical Features) =================
#
# Name:
# - همه‌ی 891 مسافر نام دارند.
# - هر نام فقط یک بار آمده، بنابراین خود ستون Name به تنهایی احتمالاً ویژگی مناسبی نیست.
# - اما ممکن است از داخل نام‌ها (مثل عنوان Mr, Mrs, Miss) ویژگی جدید استخراج کنیم.
#
# Sex:
# - هیچ مقدار گمشده‌ای ندارد.
# - فقط دو مقدار دارد: male و female.
# - تعداد مردان بیشتر از زنان است (577 مرد).
# - احتمالاً یکی از ویژگی‌های مهم برای پیش‌بینی Survival است.
#
# Ticket:
# - همه‌ی مسافران شماره بلیت دارند.
# - 681 شماره بلیت متفاوت وجود دارد، یعنی بعضی افراد بلیت مشترک داشته‌اند.
# - اشتراک بلیت ممکن است نشان‌دهنده‌ی خانواده یا گروه سفر باشد و شاید اطلاعات مفیدی داشته باشد.
#
# Cabin:
# - تعداد زیادی مقدار گمشده دارد (فقط 204 مقدار ثبت شده).
# - به دلیل کمبود شدید اطلاعات، باید بررسی کنیم که آیا نگه داشتن آن ارزش دارد یا نه.
# - ممکن است از حرف اول Cabin (Deck) بتوان ویژگی جدید ساخت.
#
# Embarked:
# - فقط 2 مقدار گمشده دارد.
# - سه مقدار دارد:
#   C = Cherbourg
#   Q = Queenstown
#   S = Southampton
# - چون تعداد مقادیر گمشده کم است، به راحتی قابل جایگزینی است.
#
# =============================================================================

In [11]:
#الان دقیقاً در مرحله EDA (Exploratory Data Analysis) هستیم
#میخواهیم ببینیم کدام ویژگی‌ها با هدف (Survived) ارتباط بیشتری دارند.
#Correlation  یعنی یک معیار آماری که نشان می‌دهد دو متغیر چقدر با هم تغییر می‌کنند. فیچرها با ریزالت 
#اول برای ستون‌های عددی correlation می‌گیریم:
df.corr(numeric_only=True)["Survived"].sort_values(ascending=False)#اگه ستون survuved رو مشخص نکرده بودم کورولیشن همه ستون ها نسبت به هم رو میداد

Survived       1.000000
Fare           0.257307
Parch          0.081629
PassengerId   -0.005007
SibSp         -0.035322
Age           -0.077221
Pclass        -0.338481
Name: Survived, dtype: float64

In [12]:
#این اعداد نشان می‌دهند هر ویژگی عددی چقدر با Survived رابطه خطی دارد.
#Survived       1.000000 : ینی هر ستون با خودش correlation برابر ۱ دارد، پس مهم نیست

#Fare یک رابطه مثبت متوسطِ ضعیف دارد. ینی مسافرانی که کرایه بیشتری پرداخت کرده‌اند، بیشتر نجات پیدا کرده‌اند.

#Parch تعداد والدین یا فرزندان همراه مسافر. 
#Correlation پایین است، یعنی فقط با این عدد نمی‌توانیم چیز زیادی بفهمیم
#اما شاید ترکیبش با SibSp مفید باشد (بعداً FamilySize می‌سازیم).

#PassengerId تقریباً صفر. یعنی شماره مسافر هیچ رابطه‌ای با زنده ماندن ندارد. این ستون معمولاً باید حذف شود: چون فقط یک شناسه است، نه اطلاعات واقعی.


#Age ضعیف است. 
#یعنی سن به شکل خطی با نجات ارتباط ندارد. البته ممکنه رابطه غیر خطی داشته باشند ولی کورولیشن این رو نمیبینه	
#مثال رابطه غیر خطی:
# 0-10 سال  → شانس بالا
#20-50 سال → متوسط
#70+ سال   → پایین

#Pclass قوی‌ترین رابطه عددی ماست. منفی بودنش به خاطر نحوه شماره‌گذاری است: 
#1 = First Class
#2 = Second Class
#3 = Third Class 
#Pclass بیشتر
      #↓
#کلاس پایین‌تر
      #↓
#شانس بقا کمتر

In [13]:
pd.crosstab(df["Pclass"], df["Survived"], normalize="index")

Survived,0,1
Pclass,,
1,0.370370,0.629630
2,0.527174,0.472826
3,0.757637,0.242363


In [14]:
#Pclass و Fare احتمالاً اطلاعات مشابهی دارند.چون کلاس بالاتر کرایه بیشتر پس در مدل، هر دو می‌توانند مفید باشند.

In [15]:
#فیچرهای عیر عددی رو با کراس تب بررسی میکنن

# که برویم سراغ رابطه Sex با Survived:
pd.crosstab(df["Sex"], df["Survived"], normalize="index") #یعنی نرمال‌سازی (Normalize کردن) ؛ یعنی مقادیر را به یک مقیاس قابل مقایسه تبدیل کنیم
#ینی نرمال‌سازی را بر اساس ردیف‌ها انجام بده.یعنی هر ردیف را جداگانه جمع کن و تقسیم کن normalize="index"
#normalize="columns" = درصدگیری داخل هر ستون

Survived,0,1
Sex,,
female,0.257962,0.742038
male,0.811092,0.188908


In [16]:
#قدم بعدی که ارزش زیادی دارد این است که Sex و Pclass را با هم ترکیب کنیم، چون در حادثه تایتانیک این دو عامل با هم خیلی اثر داشتند.
pd.crosstab([df["Pclass"], df["Sex"]], df["Survived"], normalize="index") #این به ما میگه زنان و مردان هر کلاس چند درصد نجات پیدا کردن

Survived              0         1
Pclass Sex                       
1      female  0.031915  0.968085
       male    0.631148  0.368852
2      female  0.078947  0.921053
       male    0.842593  0.157407
3      female  0.500000  0.500000
       male    0.864553  0.135447

In [17]:
#از دید ماشین لرنینگ این نشان می‌دهد که احتمالاً
#Sex یک فیچر بسیار مهم است.
#Pclass یک فیچر بسیار مهم است.
#اینتراکشن بین آنها هم مهم است
#در مدل‌های درختی مثل Random Forest، این تعامل‌ها معمولاً خودکار کشف می‌شوند.


In [18]:
#چیز دیگه ای که میخوام بررسی کنم 
#چون تعداد اعضای خانواده همراه هم روی شانس بقا اثر داشته.

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1 #عدد یک واسه اینه که خود شخص رو هم بشماریم

#حالا که فیچر جدید رو ساختیم میخوایم ببینیم روی نجات اثر داشته
pd.crosstab(df["FamilySize"], df["Survived"], normalize="index")

Survived,0,1
FamilySize,,
1,0.696462,0.303538
2,0.447205,0.552795
3,0.421569,0.578431
4,0.275862,0.724138
5,0.800000,0.200000
6,0.863636,0.136364
7,0.666667,0.333333
8,1.000000,0.000000
11,1.000000,0.000000


In [19]:
#این نتیجه نشون میده تعداد اعضای خانواده روی بقا اثر داره ولی رابطه خطی نداره با بقا

In [20]:
pd.crosstab(df["Embarked"], df["Survived"], normalize="index") #این به ما می‌گوید:در هر بندر، چند درصد مردم زنده ماندند؟

Survived,0,1
Embarked,,
C,0.446429,0.553571
Q,0.610390,0.389610
S,0.663043,0.336957


In [21]:
# - مسافرانی که از بندر Cherbourg (C) سوار شده‌اند، بیشترین نرخ بقا را داشته‌اند (~55%).
# این به معنای آن نیست که بندر سوار شدن باعث نجات شده است.
# احتمالاً Embarked با ویژگی‌های دیگری مانند Pclass و Fare مرتبط است؛
# بنابراین این فقط یک رابطه (Association) است، نه رابطه علت و معلولی (Causation).

In [22]:
#EDA تموم شد 
#تو نوت بوک بعدی میرم سراع Data Preprocessing
#یعنی داده را برای مدل آماده می‌کنیم.
#مدیریت مقادیر گمشده (Missing Values)
#Age
#Embarked
#Cabin
#تبدیل داده‌های متنی به عدد
#Sex
#Embarked
#انتخاب ستون‌هایی که وارد مدل می‌شوند
#PassengerId حذف
#بررسی Name
#بررسی Ticket
#تصمیم درباره Cabin
#ساخت X و y
#تقسیم داده به Train و Test
#آموزش اولین مدل